# 📈 Portfolio Holdings Analysis Report
## 보유기업 분석 보고서 — 펀더멘탈 트래킹 & 매도전략

| 셀 | 역할 |
|----|------|
| 1  | 경로 자동 감지 |
| 2  | Import & DB 연결 |
| 3  | **보유 포트폴리오 입력** ← 여기만 수정 |
| 4  | 최신 Valuation 로드 |
| 5  | Valuation 변화 추적 (월별 time-series) |
| 6  | 매출 추정치 변화 시각화 |
| 7  | 매도 신호 판단 (Upside 기준 + 손절) |
| 8  | 기업별 종합 대시보드 |
| 9  | 포트폴리오 요약 리포트 저장 |


## Cell 1 · 경로 자동 감지

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_CANDIDATE_ROOTS = [
    r'C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast',
    r'C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy',
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / 'DATA').is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f'[PATH] 자동 감지 성공 : {root}')
            return root
    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, 'DATA')):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f'[PATH] 후보 경로 사용 : {candidate}')
            return candidate
    raise EnvironmentError('DATA 폴더를 찾을 수 없습니다.')

_ROOT = _setup_path()
print(f'[확인] 프로젝트 루트 : {_ROOT}')

## Cell 2 · Import & DB 연결

In [ ]:
import gc, math, warnings
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import pymysql
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from IPython.display import display, HTML
from sqlalchemy import text

matplotlib.rcParams['font.family']       = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.dpi']         = 120

from DATA.config import get_db_info, get_engine

# Screened Ticker List (섹터 정보용)
try:
    from DATA.us_target_ticker_list_screened_20260411 import ticker_sector_map as US_SECTOR_MAP
except ImportError:
    import glob, importlib.util
    candidates = glob.glob(os.path.join(_ROOT, 'DATA', 'us_target_ticker_list_screened*.py'))
    if candidates:
        spec = importlib.util.spec_from_file_location('ticker_mod', candidates[-1])
        mod  = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        US_SECTOR_MAP = getattr(mod, 'ticker_sector_map', {})
    else:
        US_SECTOR_MAP = {}

db_info = get_db_info()
engine  = get_engine(db_info)

def get_conn():
    return pymysql.connect(
        host       = db_info['host'],
        port       = int(db_info.get('port', 3307)),
        user       = db_info['user'],
        password   = db_info['password'],
        db         = db_info.get('database', 'investar'),
        charset    = 'utf8mb4',
        autocommit = False,
        cursorclass= pymysql.cursors.DictCursor,
    )

try:
    with engine.connect() as c:
        c.execute(text('SELECT 1'))
    print(f'[OK] DB 연결 성공  host={db_info["host"]}:{db_info["port"]}')
except Exception as e:
    print(f'[FAIL] DB 연결 실패: {e}')

## Cell 3 · 보유 포트폴리오 입력  ← **여기만 수정**

매월 평가 시 이 셀만 업데이트하면 됩니다.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  보유 포트폴리오 — 매월 업데이트
#  매수가(buy_price), 보유 수량(shares), 매수일(buy_date) 입력
# ══════════════════════════════════════════════════════════════
PORTFOLIO = [
    # 예시 — 실제 보유 내역으로 교체하세요
    {'ticker': 'NVDA', 'buy_price': 850.0,  'shares': 10, 'buy_date': '2024-01-15'},
    {'ticker': 'MSFT', 'buy_price': 380.0,  'shares':  5, 'buy_date': '2024-02-01'},
    {'ticker': 'AAPL', 'buy_price': 175.0,  'shares':  8, 'buy_date': '2024-03-10'},
    {'ticker': 'LLY',  'buy_price': 750.0,  'shares':  3, 'buy_date': '2024-06-20'},
    {'ticker': 'ANET', 'buy_price': 230.0,  'shares': 12, 'buy_date': '2024-09-05'},
]

# ── 매도 전략 파라미터 ───────────────────────────────────────
SELL_STOPLOSE      = -0.20   # 손절매: 매수가 대비 -20%
SELL_REDUCE_UPSIDE =  0.10   # 비중축소 신호: Upside 10% 이하
SELL_ALL_UPSIDE    =  0.00   # 전량매도 신호: Upside 0% 이하 (목표가 도달)

# ── 평가 파라미터 ────────────────────────────────────────────
EVAL_DATE      = datetime.today().strftime('%Y-%m-%d')
HISTORY_MONTHS = 12    # Valuation 추적 기간 (개월)
REVENUE_MODEL  = 'Ensemble'
REVENUE_ITEM   = 'sale'

TBL_FCFF    = 'us_fcff_dcf_valuation'
TBL_RELV    = 'us_relative_valuation'
TBL_REVENUE = 'us_revenue_forecast_data'

REPORT_DIR  = os.path.join(_ROOT, 'reports')
os.makedirs(REPORT_DIR, exist_ok=True)
REPORT_FILE = os.path.join(REPORT_DIR, f'portfolio_report_{EVAL_DATE}.xlsx')

PORT_TICKERS = [p['ticker'] for p in PORTFOLIO]
PORT_DF      = pd.DataFrame(PORTFOLIO)

print(f'[OK] 보유 종목 {len(PORT_TICKERS)}개: {PORT_TICKERS}')
print(f'[OK] 평가일: {EVAL_DATE}')
display(PORT_DF)

## Cell 4 · 최신 Valuation 로드

In [ ]:
def load_latest_valuation(tickers: List[str]) -> pd.DataFrame:
    """FCFF + Relative 최신 결과 조인"""
    ticker_str = "','".join(tickers)

    sql_fcff = f"""
        SELECT f.ticker, f.date AS date_fcff,
               f.target_price AS tp_fcff, f.current_price, f.upside_pct AS upside_fcff,
               f.roic, f.discount_rate AS wacc, f.g_terminal,
               f.opm_forecast, f.tax_rate, f.reinvestment_rate, f.fcff,
               f.enterprise_value, f.net_debt, f.equity_value, f.shares
        FROM {TBL_FCFF} f
        INNER JOIN (
            SELECT ticker, MAX(date) AS max_date
            FROM {TBL_FCFF} WHERE ticker IN ('{ticker_str}')
            GROUP BY ticker
        ) m ON f.ticker=m.ticker AND f.date=m.max_date
    """
    sql_relv = f"""
        SELECT r.ticker, r.date AS date_relv,
               r.tp_avg AS tp_relv, r.upside_avg AS upside_relv,
               r.roe_y1, r.roe_y2, r.g_est, r.re_mid, r.beta_ensemble,
               r.pbr_theory, r.per_theory, r.psr_theory,
               r.tp_pbr, r.tp_per, r.tp_psr, r.sector
        FROM {TBL_RELV} r
        INNER JOIN (
            SELECT ticker, MAX(date) AS max_date
            FROM {TBL_RELV} WHERE ticker IN ('{ticker_str}')
            GROUP BY ticker
        ) m ON r.ticker=m.ticker AND r.date=m.max_date
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql_fcff)
            fcff_rows = cur.fetchall()
            cur.execute(sql_relv)
            relv_rows = cur.fetchall()
    finally:
        conn.close()

    df_f = pd.DataFrame(fcff_rows)
    df_r = pd.DataFrame(relv_rows)

    if df_f.empty and df_r.empty:
        return pd.DataFrame()
    elif df_f.empty:
        return df_r
    elif df_r.empty:
        return df_f

    df = pd.merge(df_f, df_r, on='ticker', how='outer')
    # 가중 적정주가
    df['tp_weighted'] = (
        df['tp_fcff'].fillna(0) * 0.60 +
        df['tp_relv'].fillna(0) * 0.40
    ).where(df['tp_fcff'].notna() | df['tp_relv'].notna())
    df['current_price'] = df['current_price'].fillna(df.get('current_price', None))
    df['upside_weighted'] = (
        (df['tp_weighted'] - df['current_price']) / df['current_price'] * 100
    )
    return df


df_latest = load_latest_valuation(PORT_TICKERS)

# 포트폴리오 정보 합치기
if not df_latest.empty:
    df_port = pd.merge(PORT_DF, df_latest, on='ticker', how='left')
    # 현재 수익률
    df_port['return_pct'] = (
        (df_port['current_price'] - df_port['buy_price']) / df_port['buy_price'] * 100
    )
    df_port['market_value'] = df_port['current_price'] * df_port['shares']
    df_port['book_value']   = df_port['buy_price']     * df_port['shares']
    df_port['pnl']          = df_port['market_value']  - df_port['book_value']
else:
    df_port = PORT_DF.copy()
    print('[WARN] Valuation 데이터 없음 — DB 계산 완료 후 재실행')

print(f'[OK] 포트폴리오 Valuation 로드 완료')
show_cols = ['ticker','buy_price','current_price','return_pct',
             'tp_weighted','upside_weighted','sector']
show_cols = [c for c in show_cols if c in df_port.columns]
display(df_port[show_cols])

## Cell 5 · Valuation 변화 추적 (월별 time-series)

In [ ]:
def load_valuation_history(tickers: List[str], months: int = HISTORY_MONTHS) -> Dict:
    """
    FCFF + Relative 테이블에서 최근 N개월 월별 적정주가 변화 추출
    Returns: {ticker: {'fcff': df, 'relv': df}}
    """
    since = (datetime.today() - timedelta(days=months*31)).strftime('%Y-%m-%d')
    ticker_str = "','".join(tickers)

    sql_fcff = f"""
        SELECT ticker, date, target_price AS tp_fcff,
               current_price, upside_pct AS upside_fcff,
               roic, discount_rate AS wacc, g_terminal
        FROM {TBL_FCFF}
        WHERE ticker IN ('{ticker_str}') AND date >= '{since}'
        ORDER BY ticker, date
    """
    sql_relv = f"""
        SELECT ticker, date, tp_avg AS tp_relv,
               upside_avg AS upside_relv,
               roe_y2, g_est, re_mid
        FROM {TBL_RELV}
        WHERE ticker IN ('{ticker_str}') AND date >= '{since}'
        ORDER BY ticker, date
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql_fcff)
            fcff_rows = cur.fetchall()
            cur.execute(sql_relv)
            relv_rows = cur.fetchall()
    finally:
        conn.close()

    df_f = pd.DataFrame(fcff_rows)
    df_r = pd.DataFrame(relv_rows)

    history = {}
    for tkr in tickers:
        history[tkr] = {
            'fcff': df_f[df_f['ticker']==tkr].copy() if not df_f.empty else pd.DataFrame(),
            'relv': df_r[df_r['ticker']==tkr].copy() if not df_r.empty else pd.DataFrame(),
        }
    return history


val_history = load_valuation_history(PORT_TICKERS)

# ── 적정주가 변화 추이 시각화 ─────────────────────────────────
n_tickers = len(PORT_TICKERS)
fig, axes = plt.subplots(
    n_tickers, 1,
    figsize=(14, n_tickers * 4),
    squeeze=False
)

for idx, tkr in enumerate(PORT_TICKERS):
    ax = axes[idx][0]
    hist = val_history.get(tkr, {})

    df_fh = hist.get('fcff', pd.DataFrame())
    df_rh = hist.get('relv', pd.DataFrame())

    has_data = False

    if not df_fh.empty and 'tp_fcff' in df_fh.columns:
        df_fh['date'] = pd.to_datetime(df_fh['date'])
        ax.plot(df_fh['date'], df_fh['tp_fcff'], 'o-', color='#3498DB',
                linewidth=2, markersize=5, label='FCFF 적정주가')
        if 'current_price' in df_fh.columns:
            ax.plot(df_fh['date'], df_fh['current_price'], '--', color='#2C3E50',
                    linewidth=1.5, label='현재주가')
        has_data = True

    if not df_rh.empty and 'tp_relv' in df_rh.columns:
        df_rh['date'] = pd.to_datetime(df_rh['date'])
        ax.plot(df_rh['date'], df_rh['tp_relv'], 's--', color='#E74C3C',
                linewidth=2, markersize=5, label='Relative 적정주가')
        has_data = True

    # 매수가 수평선
    buy_price = PORT_DF[PORT_DF['ticker']==tkr]['buy_price'].values
    if len(buy_price) > 0:
        ax.axhline(buy_price[0], color='green', linestyle=':', linewidth=1.5,
                   label=f'매수가 ${buy_price[0]:.0f}')
        ax.axhline(buy_price[0] * (1 + SELL_STOPLOSE), color='red',
                   linestyle=':', linewidth=1.2,
                   label=f'손절선 ${buy_price[0]*(1+SELL_STOPLOSE):.0f}')

    sector = US_SECTOR_MAP.get(tkr, {}).get('sector', 'N/A')
    ax.set_title(f'{tkr}  [{sector}]  — 적정주가 추이', fontsize=12, fontweight='bold')
    ax.set_ylabel('주가 ($)')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)

    if not has_data:
        ax.text(0.5, 0.5, f'{tkr}: Valuation 데이터 없음',
                ha='center', va='center', transform=ax.transAxes, fontsize=12)

plt.suptitle(f'보유 종목 Valuation 추이  ({EVAL_DATE})', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Cell 6 · 매출 추정치 변화 시각화

In [ ]:
def load_revenue_history(ticker: str, model: str = REVENUE_MODEL) -> Dict:
    """특정 ticker의 실제 + 예측 매출 데이터 (최신 forecast_date 기준)"""
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            # 최신 forecast_date
            cur.execute(f"""
                SELECT MAX(forecast_date) AS fd
                FROM {TBL_REVENUE}
                WHERE ticker=%s AND item=%s
            """, (ticker, REVENUE_ITEM))
            row = cur.fetchone()
            if not row or not row['fd']:
                return {}
            fd = str(row['fd'])

            # 직전 forecast_date (전월 비교용)
            cur.execute(f"""
                SELECT MAX(forecast_date) AS fd_prev
                FROM {TBL_REVENUE}
                WHERE ticker=%s AND item=%s AND forecast_date < %s
            """, (ticker, REVENUE_ITEM, fd))
            prev_row = cur.fetchone()
            fd_prev  = str(prev_row['fd_prev']) if prev_row and prev_row['fd_prev'] else None

            # 실제값
            cur.execute(f"""
                SELECT date, value FROM {TBL_REVENUE}
                WHERE ticker=%s AND item=%s AND model='actual' AND forecast_date=%s
                ORDER BY date
            """, (ticker, REVENUE_ITEM, fd))
            act_rows = cur.fetchall()

            # 최신 예측값
            cur.execute(f"""
                SELECT date, value FROM {TBL_REVENUE}
                WHERE ticker=%s AND item=%s AND model=%s
                  AND data_type='forecast' AND forecast_date=%s
                ORDER BY date
            """, (ticker, REVENUE_ITEM, model, fd))
            fct_rows = cur.fetchall()

            # 전월 예측값
            fct_prev_rows = []
            if fd_prev:
                cur.execute(f"""
                    SELECT date, value FROM {TBL_REVENUE}
                    WHERE ticker=%s AND item=%s AND model=%s
                      AND data_type='forecast' AND forecast_date=%s
                    ORDER BY date
                """, (ticker, REVENUE_ITEM, model, fd_prev))
                fct_prev_rows = cur.fetchall()
    finally:
        conn.close()

    return {
        'ticker'    : ticker,
        'fd'        : fd,
        'fd_prev'   : fd_prev,
        'actual'    : pd.DataFrame(act_rows),
        'forecast'  : pd.DataFrame(fct_rows),
        'fcst_prev' : pd.DataFrame(fct_prev_rows),
    }


# ── 전 종목 매출 추이 시각화 ──────────────────────────────────
n_tickers = len(PORT_TICKERS)
fig, axes = plt.subplots(n_tickers, 1, figsize=(14, n_tickers * 4), squeeze=False)

for idx, tkr in enumerate(PORT_TICKERS):
    ax = axes[idx][0]
    rev = load_revenue_history(tkr)

    if not rev or rev.get('actual', pd.DataFrame()).empty:
        ax.text(0.5, 0.5, f'{tkr}: Revenue 데이터 없음',
                ha='center', va='center', transform=ax.transAxes)
        continue

    act = rev['actual'].copy()
    fct = rev['forecast'].copy()
    fct_prev = rev.get('fcst_prev', pd.DataFrame())

    act['date'] = pd.to_datetime(act['date'])
    # 최근 12분기만 표시
    act = act.sort_values('date').tail(12)

    # 단위 자동 조정
    max_val = max(act['value'].max(), fct['value'].max() if not fct.empty else 0)
    if max_val >= 1e9:
        div, unit = 1e9, 'B'
    elif max_val >= 1e6:
        div, unit = 1e6, 'M'
    else:
        div, unit = 1, ''

    ax.bar(act['date'], act['value']/div, width=60,
           color='#2C3E50', alpha=0.8, label='실적')

    if not fct.empty:
        fct['date'] = pd.to_datetime(fct['date'])
        fct = fct.sort_values('date')
        ax.bar(fct['date'], fct['value']/div, width=60,
               color='#E74C3C', alpha=0.6, label=f'예측 ({rev["fd"][:7]})')

    if not fct_prev.empty:
        fct_prev['date'] = pd.to_datetime(fct_prev['date'])
        fct_prev = fct_prev.sort_values('date')
        ax.plot(fct_prev['date'], fct_prev['value']/div, 'o--',
                color='#F39C12', linewidth=1.5, markersize=4,
                label=f'전월 예측 ({rev["fd_prev"][:7] if rev["fd_prev"] else "N/A"})')

    # 성장률 계산 & 표시
    if not fct.empty and len(act) >= 4 and len(fct) >= 4:
        g4 = (sum(fct['value'].tolist()[:4]) / sum(act['value'].tolist()[-4:]) - 1) * 100
        ax.set_title(f'{tkr}  매출 추이  (4Q 예상 성장률: {g4:+.1f}%)', fontsize=12, fontweight='bold')
    else:
        ax.set_title(f'{tkr}  매출 추이', fontsize=12, fontweight='bold')

    ax.set_ylabel(f'매출 ({unit}$)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'보유 종목 매출 추이 및 예측 비교  ({EVAL_DATE})', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Cell 7 · 매도 신호 판단

In [ ]:
# ──────────────────────────────────────────────────────────────
#  매도 신호 판단 로직
#  1) 손절선: 현재가 < 매수가 × (1 - 20%) → STOP LOSS
#  2) 비중축소: Upside ≤ 10% → REDUCE 50%
#  3) 전량매도: Upside ≤ 0%  → SELL ALL
#  4) 보유유지: 위 조건 미해당
# ──────────────────────────────────────────────────────────────

def sell_signal(row) -> Dict:
    tkr        = row['ticker']
    buy_price  = row.get('buy_price', None)
    curr_price = row.get('current_price', None)
    upside     = row.get('upside_weighted', None)   # %

    signal = 'HOLD'
    reason = ''

    if curr_price is None or pd.isna(curr_price):
        return {'signal': 'DATA_MISSING', 'reason': '현재가 없음', 'priority': 0}

    # 손절
    if buy_price and curr_price <= buy_price * (1 + SELL_STOPLOSE):
        signal = '🔴 STOP LOSS'
        reason = f'현재가 ${curr_price:.2f} ≤ 손절선 ${buy_price*(1+SELL_STOPLOSE):.2f}'
        return {'signal': signal, 'reason': reason, 'priority': 4}

    if upside is not None and not pd.isna(upside):
        upside_decimal = upside / 100
        if upside_decimal <= SELL_ALL_UPSIDE:
            signal = '🔴 SELL ALL'
            reason = f'Upside {upside:.1f}% — 목표가 도달'
            return {'signal': signal, 'reason': reason, 'priority': 3}
        elif upside_decimal <= SELL_REDUCE_UPSIDE:
            signal = '🟡 REDUCE 50%'
            reason = f'Upside {upside:.1f}% ≤ {SELL_REDUCE_UPSIDE*100:.0f}% 기준선'
            return {'signal': signal, 'reason': reason, 'priority': 2}
        else:
            signal = '🟢 HOLD'
            reason = f'Upside {upside:.1f}% — 충분한 상승여력'
            return {'signal': signal, 'reason': reason, 'priority': 1}

    return {'signal': '⚪ HOLD (No Val.)', 'reason': 'Valuation 미계산', 'priority': 0}


if not df_port.empty:
    signal_results = df_port.apply(sell_signal, axis=1)
    df_port['signal']   = signal_results.apply(lambda x: x['signal'])
    df_port['sig_reason'] = signal_results.apply(lambda x: x['reason'])

    # 포트폴리오 요약
    total_book   = df_port['book_value'].sum()   if 'book_value'   in df_port.columns else 0
    total_market = df_port['market_value'].sum() if 'market_value' in df_port.columns else 0
    total_pnl    = df_port['pnl'].sum()          if 'pnl'          in df_port.columns else 0

    print('=' * 70)
    print(f'  포트폴리오 매도 신호 분석  ({EVAL_DATE})')
    print('=' * 70)

    display_cols = ['ticker','buy_price','current_price','return_pct',
                    'upside_weighted','signal','sig_reason']
    display_cols = [c for c in display_cols if c in df_port.columns]
    display(df_port[display_cols].sort_values('signal'))

    print(f'\n  총 투자금액 : ${total_book:,.0f}')
    print(f'  현재 평가액 : ${total_market:,.0f}')
    print(f'  평가 손익   : ${total_pnl:+,.0f}  ({total_pnl/total_book*100:+.1f}%)')
else:
    print('[WARN] 포트폴리오 데이터 없음')

## Cell 8 · 기업별 종합 대시보드

In [ ]:
# ──────────────────────────────────────────────────────────────
# 기업별 2×2 대시보드
#  [좌상] 적정주가 추이 (FCFF / Relative)
#  [우상] Upside % 추이
#  [좌하] 매출 실적 + 예측
#  [우하] 핵심 지표 테이블 (ROIC, ROE, WACC, g, upside)
# ──────────────────────────────────────────────────────────────

def plot_company_dashboard(ticker: str):
    hist = val_history.get(ticker, {})
    rev  = load_revenue_history(ticker)
    port_row = PORT_DF[PORT_DF['ticker']==ticker].iloc[0] if len(PORT_DF[PORT_DF['ticker']==ticker]) else {}

    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])  # 적정주가 추이
    ax2 = fig.add_subplot(gs[0, 1])  # Upside % 추이
    ax3 = fig.add_subplot(gs[1, 0])  # 매출 추이
    ax4 = fig.add_subplot(gs[1, 1])  # 핵심 지표

    sector   = US_SECTOR_MAP.get(ticker, {}).get('sector', 'N/A')
    mktcap   = US_SECTOR_MAP.get(ticker, {}).get('mktCap_B', 0)
    fig.suptitle(f'{ticker}  [{sector}]  시총 ${mktcap:.1f}B  —  {EVAL_DATE}',
                 fontsize=14, fontweight='bold')

    # ── [좌상] 적정주가 추이 ──────────────────────────────────
    df_fh = hist.get('fcff', pd.DataFrame())
    df_rh = hist.get('relv', pd.DataFrame())

    if not df_fh.empty and 'tp_fcff' in df_fh.columns:
        df_fh['date'] = pd.to_datetime(df_fh['date'])
        ax1.plot(df_fh['date'], df_fh['tp_fcff'], 'o-', color='#3498DB',
                 lw=2, ms=6, label='FCFF 적정주가')
        if 'current_price' in df_fh.columns:
            ax1.plot(df_fh['date'], df_fh['current_price'], '--k',
                     lw=1.5, label='현재주가')

    if not df_rh.empty and 'tp_relv' in df_rh.columns:
        df_rh['date'] = pd.to_datetime(df_rh['date'])
        ax1.plot(df_rh['date'], df_rh['tp_relv'], 's--', color='#E74C3C',
                 lw=2, ms=5, label='Relative 적정주가')

    buy_p = port_row.get('buy_price', None)
    if buy_p:
        ax1.axhline(buy_p, color='green', ls=':', lw=1.5,
                    label=f'매수가 ${buy_p:.0f}')
        ax1.axhline(buy_p * (1 + SELL_STOPLOSE), color='red', ls=':', lw=1.2,
                    label=f'손절선 ${buy_p*(1+SELL_STOPLOSE):.0f}')

    ax1.set_title('적정주가 추이', fontsize=10)
    ax1.set_ylabel('주가 ($)')
    ax1.legend(fontsize=7)
    ax1.grid(True, alpha=0.3)

    # ── [우상] Upside 추이 ────────────────────────────────────
    if not df_fh.empty and 'upside_fcff' in df_fh.columns:
        ax2.plot(df_fh['date'], df_fh['upside_fcff'], 'o-', color='#3498DB',
                 lw=2, ms=5, label='FCFF Upside')
    if not df_rh.empty and 'upside_relv' in df_rh.columns:
        df_rh['date'] = pd.to_datetime(df_rh['date'])
        ax2.plot(df_rh['date'], df_rh['upside_relv'], 's--', color='#E74C3C',
                 lw=2, ms=5, label='Relative Upside')

    ax2.axhline(SELL_REDUCE_UPSIDE * 100, color='orange', ls='--', lw=1.2,
                label=f'비중축소선 {SELL_REDUCE_UPSIDE*100:.0f}%')
    ax2.axhline(30, color='gray', ls='--', lw=1.0, label='30% 기준선')
    ax2.axhline(0, color='red', ls='-', lw=0.8)
    ax2.set_title('Upside % 추이', fontsize=10)
    ax2.set_ylabel('Upside (%)')
    ax2.legend(fontsize=7)
    ax2.grid(True, alpha=0.3)

    # ── [좌하] 매출 추이 ──────────────────────────────────────
    if rev and not rev.get('actual', pd.DataFrame()).empty:
        act = rev['actual'].copy()
        fct = rev.get('forecast', pd.DataFrame())
        act['date'] = pd.to_datetime(act['date'])
        act = act.sort_values('date').tail(12)

        max_v = act['value'].max()
        div, unit = (1e9,'B') if max_v>=1e9 else (1e6,'M') if max_v>=1e6 else (1,'')

        ax3.bar(act['date'], act['value']/div, width=60,
                color='#2C3E50', alpha=0.8, label='실적')
        if not fct.empty:
            fct['date'] = pd.to_datetime(fct['date'])
            fct = fct.sort_values('date')
            ax3.bar(fct['date'], fct['value']/div, width=60,
                    color='#E74C3C', alpha=0.6, label='예측')
        ax3.set_title('매출 추이 및 예측', fontsize=10)
        ax3.set_ylabel(f'매출 ({unit}$)')
        ax3.legend(fontsize=7)
        ax3.grid(True, alpha=0.3, axis='y')
    else:
        ax3.text(0.5, 0.5, '매출 데이터 없음', ha='center', va='center',
                 transform=ax3.transAxes)

    # ── [우하] 핵심 지표 테이블 ───────────────────────────────
    ax4.axis('off')
    port_val = df_port[df_port['ticker']==ticker].iloc[0] if not df_port.empty and len(df_port[df_port['ticker']==ticker]) > 0 else {}

    def _fmt(v, fmt='.2f', default='N/A'):
        try:
            return f'{v:{fmt}}' if pd.notna(v) else default
        except:
            return default

    table_data = [
        ['지표', '값'],
        ['매수가',   f"${_fmt(port_val.get('buy_price', None), '.2f')}"],
        ['현재가',   f"${_fmt(port_val.get('current_price', None), '.2f')}"],
        ['수익률',   f"{_fmt(port_val.get('return_pct', None), '+.1f')}%"],
        ['FCFF 목표가', f"${_fmt(port_val.get('tp_fcff', None), '.2f')}"],
        ['Relative 목표가', f"${_fmt(port_val.get('tp_relv', None), '.2f')}"],
        ['가중 목표가', f"${_fmt(port_val.get('tp_weighted', None), '.2f')}"],
        ['Upside', f"{_fmt(port_val.get('upside_weighted', None), '+.1f')}%"],
        ['ROIC',  f"{_fmt(port_val.get('roic', None), '.1%')}"],
        ['WACC',  f"{_fmt(port_val.get('wacc', None), '.1%')}"],
        ['g terminal', f"{_fmt(port_val.get('g_terminal', None), '.1%')}"],
        ['ROE (y2)', f"{_fmt(port_val.get('roe_y2', None), '.1%')}"],
        ['매도 신호', str(port_val.get('signal', 'N/A'))],
    ]

    t = ax4.table(
        cellText=table_data[1:],
        colLabels=table_data[0],
        loc='center', cellLoc='center'
    )
    t.auto_set_font_size(False)
    t.set_fontsize(9)
    t.scale(1.2, 1.6)
    ax4.set_title('핵심 지표', fontsize=10)

    plt.tight_layout()
    plt.show()
    plt.close(fig)


print('기업별 대시보드 생성 중...')
for tkr in PORT_TICKERS:
    plot_company_dashboard(tkr)

## Cell 9 · 포트폴리오 요약 리포트 Excel 저장

In [ ]:
try:
    import openpyxl
    OPENPYXL_OK = True
except ImportError:
    print('[WARN] openpyxl 없음. pip install openpyxl')
    OPENPYXL_OK = False

if OPENPYXL_OK and not df_port.empty:
    writer = pd.ExcelWriter(REPORT_FILE, engine='openpyxl')

    # ── 1) 포트폴리오 요약 ────────────────────────────────────
    summary_cols = [
        'ticker','buy_date','buy_price','current_price','return_pct',
        'tp_fcff','tp_relv','tp_weighted','upside_weighted',
        'sector','signal','sig_reason','market_value','pnl'
    ]
    summary_cols = [c for c in summary_cols if c in df_port.columns]
    df_port[summary_cols].to_excel(writer, sheet_name='포트폴리오_요약', index=False)

    # ── 2) Valuation 추적 시트 (월별 변화) ───────────────────
    hist_records = []
    for tkr in PORT_TICKERS:
        hist = val_history.get(tkr, {})
        df_fh = hist.get('fcff', pd.DataFrame())
        df_rh = hist.get('relv', pd.DataFrame())
        if not df_fh.empty:
            for _, r in df_fh.iterrows():
                hist_records.append({
                    'ticker': tkr, 'date': r.get('date'), 'source': 'FCFF',
                    'tp': r.get('tp_fcff'), 'upside': r.get('upside_fcff'),
                    'roic': r.get('roic'), 'wacc': r.get('wacc'),
                    'g_terminal': r.get('g_terminal'),
                })
        if not df_rh.empty:
            for _, r in df_rh.iterrows():
                hist_records.append({
                    'ticker': tkr, 'date': r.get('date'), 'source': 'Relative',
                    'tp': r.get('tp_relv'), 'upside': r.get('upside_relv'),
                    'roe_y2': r.get('roe_y2'), 'g_est': r.get('g_est'),
                    're_mid': r.get('re_mid'),
                })

    if hist_records:
        pd.DataFrame(hist_records).to_excel(writer, sheet_name='Valuation_추적', index=False)

    # ── 3) 매출 변화 시트 ────────────────────────────────────
    rev_records = []
    for tkr in PORT_TICKERS:
        rev = load_revenue_history(tkr)
        if not rev or rev.get('actual', pd.DataFrame()).empty:
            continue
        act = rev['actual'].copy()
        act['ticker'] = tkr
        act['type']   = 'actual'
        act['fd']     = rev['fd']
        fct = rev.get('forecast', pd.DataFrame())
        if not fct.empty:
            fct = fct.copy()
            fct['ticker'] = tkr
            fct['type']   = 'forecast'
            fct['fd']     = rev['fd']
            rev_records.append(pd.concat([act, fct], ignore_index=True))
        else:
            rev_records.append(act)

    if rev_records:
        pd.concat(rev_records, ignore_index=True).to_excel(
            writer, sheet_name='매출_추이', index=False)

    writer.close()
    print(f'[OK] Excel 보고서 저장 완료: {REPORT_FILE}')
    print(f'     시트: 포트폴리오_요약 / Valuation_추적 / 매출_추이')

elif not OPENPYXL_OK:
    print('[SKIP] openpyxl 미설치')
else:
    print('[SKIP] 포트폴리오 데이터 없음')